# **Hyperparameter Tuning for the GA Meal Planner**

In [1]:
import json
import sys
from datetime import datetime, timedelta
from random import Random

from optuna import create_study
from optuna.trial import Trial
from optuna.visualization import (
    plot_optimization_history,
    plot_parallel_coordinate,
    plot_param_importances,
)
from tqdm.auto import tqdm

sys.path.append("..")

from engines import GAMealPlanner
from models import (
    Ingredient,
    MealPlanningEnvironment,
    NutritionalInformation,
    Pantry,
    PantryIngredient,
    UserPreferences,
)


The default user preferences are set for tuning the GA Meal Planner.

In [2]:
preferences = UserPreferences(
    weekly_budget=50,
    calorie_target_per_day=2500,
    protein_target_per_day=80,
    is_vegetarian=False,
    is_vegan=False,
    requires_gluten_free=False,
    requires_lactose_free=False,
)

In [3]:
all_ingredients = []

with open("../recipe_extraction/supplemented_structured_ingredients.json", "r") as f:
    ingredients_data = json.load(f)

    for ingredient_data in ingredients_data:
        ingredient_nutritional_information = NutritionalInformation(**ingredient_data["nutritional_information"])
        ingredient = Ingredient(
            name=ingredient_data["name"],
            nutritional_information=ingredient_nutritional_information,
        )
        all_ingredients.append(ingredient)


def get_ingredient(ingredient_name: str) -> Ingredient | None:
    return next((i for i in all_ingredients if i.name == ingredient_name), None)


def get_pantry_ingredient(
    ingredient_name: str, estimated_expiration_date: datetime, estimated_financial_cost: float
) -> PantryIngredient:
    ingredient = get_ingredient(ingredient_name)

    if ingredient is None:
        raise ValueError(f"Ingredient '{ingredient_name}' not found in ingredient list")

    return PantryIngredient(
        name=ingredient.name,
        nutritional_information=ingredient.nutritional_information,
        estimated_expiration_date=estimated_expiration_date,
        estimated_financial_cost=estimated_financial_cost,
    )

In [4]:
PANTRY_SEED = 1
PANTRY_SIZE = 20

QUANTITY_MIN = 200
QUANTITY_MAX = 2000

COST_MIN = 0.50
COST_MAX = 5.00

EXPIRY_DAYS_MIN = 1
EXPIRY_DAYS_MAX = 14

In [5]:
rng = Random(PANTRY_SEED)
CURRENT_DATE = datetime.now()

sampled_ingredients = rng.sample(all_ingredients, PANTRY_SIZE)

pantry = Pantry()

for ingredient in sampled_ingredients:
    quantity = rng.randint(QUANTITY_MIN, QUANTITY_MAX)
    cost = round(rng.uniform(COST_MIN, COST_MAX), 2)
    expiry_days = rng.randint(EXPIRY_DAYS_MIN, EXPIRY_DAYS_MAX)

    pantry_ingredient = PantryIngredient(
        name=ingredient.name,
        nutritional_information=ingredient.nutritional_information,
        estimated_expiration_date=CURRENT_DATE + timedelta(days=expiry_days),
        estimated_financial_cost=cost,
    )
    pantry.add(pantry_ingredient, quantity)

meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

print(f"Loaded {len(meal_planning_environment.recipes)} recipes")
print(f"\nPantry ({PANTRY_SIZE} ingredients, seed = {PANTRY_SEED}):")
pantry.print()

Loaded 19716 recipes

Pantry (20 ingredients, seed = 1):
---
Quantity: 1911 g
Ingredient: unsalted kosher-for-Passover pareve (non-dairy) margarine
	Estimated Expiration Date: 2026-05-06
	Estimated Financial Cost per 100g: EUR 2.25
	Nutritional Information:
		Calories: 714.0 kcal
		Carbohydrates: 0.0 g
		Sugar: None g
		Protein: 0.0 g
		Fat: 78.57 g
		Saturated Fat: 10.71 g
		Fiber: None g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1761 g
Ingredient: white truffles
	Estimated Expiration Date: 2026-05-08
	Estimated Financial Cost per 100g: EUR 3.95
	Nutritional Information:
		Calories: 639.0 kcal
		Carbohydrates: 38.89 g
		Sugar: 38.89 g
		Protein: 5.56 g
		Fat: 52.78 g
		Saturated Fat: 36.11 g
		Fiber: 0.0 g
		Sodium: 83.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1112 g
Ingredient: orange wheel
	Estimated Expiration Date: 2026-05-09
	Estimated Financial Cost per 100g: EUR 1.7

In [6]:
study = create_study(direction="maximize", study_name="GA Meal Planner Hyperparameter Tuning")

[I 2026-04-26 15:57:38,695] A new study created in memory with name: GA Meal Planner Hyperparameter Tuning


In [7]:
NUM_DAYS = 7
NUM_MEALS_PER_DAY = 3
TUNING_SEED = 1

In [8]:
def optimise(trial: Trial) -> float:
    """
    Optimisation objective function for the GA Meal Planner.

    Tunes both the GA hyperparameters and the fitness function weights.

    :param trial: an Optuna trial object
    :type trial: optuna.trial.Trial

    :return: the fitness score of the best meal plan generated by the GA Meal Planner with the given hyperparameters
    :rtype: float
    """

    num_generations = trial.suggest_int("num_generations", 20, 140, step=20)
    population_size = trial.suggest_int("population_size", 20, 200, step=20)
    num_parents_mating = trial.suggest_int("num_parents_mating", 2, population_size, step=2)
    parent_selection_type = trial.suggest_categorical(
        "parent_selection_type", ["sss", "rws", "sus", "random", "tournament"]
    )
    K_tournament = trial.suggest_int("K_tournament", 2, 10)
    keep_elitism = trial.suggest_int("keep_elitism", 0, 1)
    crossover_type = trial.suggest_categorical("crossover_type", ["single_point", "two_points", "uniform", "scattered"])
    crossover_probability = trial.suggest_float("crossover_probability", 0.5, 1.0, step=0.1)
    mutation_type = trial.suggest_categorical("mutation_type", ["random", "swap", "inversion", "scramble"])
    mutation_probability = trial.suggest_float("mutation_probability", 0.01, 0.5, step=0.01)

    ga_meal_planner = GAMealPlanner(meal_planning_environment)

    _, best_fitness = ga_meal_planner.generate_meal_plan(
        num_days=NUM_DAYS,
        meals_per_day=NUM_MEALS_PER_DAY,
        num_generations=num_generations,
        num_parents_mating=num_parents_mating,
        population_size=population_size,
        parent_selection_type=parent_selection_type,
        K_tournament=K_tournament,
        keep_elitism=keep_elitism,
        crossover_type=crossover_type,
        crossover_probability=crossover_probability,
        mutation_type=mutation_type,
        mutation_probability=mutation_probability,
        generation_print_interval=None,
        seed=TUNING_SEED,
    )

    return best_fitness

In [9]:
NUM_TRIALS = 100

In [10]:
with tqdm(total=NUM_TRIALS, desc="Optuna trials", unit="trial") as pbar:

    def _tqdm_callback(study, trial):
        pbar.update(1)
        pbar.set_postfix(best=f"{study.best_value:.4f}", trial=trial.number)

    study.optimize(optimise, n_trials=NUM_TRIALS, callbacks=[_tqdm_callback])

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best fitness score: {study.best_value:.4f}")

Optuna trials:   0%|          | 0/100 [00:00<?, ?trial/s]

[I 2026-04-26 15:58:29,983] Trial 0 finished with value: -159.44404379311007 and parameters: {'num_generations': 20, 'population_size': 160, 'num_parents_mating': 42, 'parent_selection_type': 'sus', 'K_tournament': 2, 'keep_elitism': 1, 'crossover_type': 'scattered', 'crossover_probability': 0.9, 'mutation_type': 'random', 'mutation_probability': 0.31}. Best is trial 0 with value: -159.44404379311007.
[I 2026-04-26 15:58:32,968] Trial 1 finished with value: -91.71477435666868 and parameters: {'num_generations': 60, 'population_size': 180, 'num_parents_mating': 146, 'parent_selection_type': 'sss', 'K_tournament': 9, 'keep_elitism': 0, 'crossover_type': 'two_points', 'crossover_probability': 0.6, 'mutation_type': 'scramble', 'mutation_probability': 0.27}. Best is trial 1 with value: -91.71477435666868.
[I 2026-04-26 15:59:19,655] Trial 2 finished with value: -131.02014670456106 and parameters: {'num_generations': 140, 'population_size': 120, 'num_parents_mating': 98, 'parent_selection_ty

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params

print("Best hyperparameters:")
for key, value in best_params.items():
    print(f"\t- {key}: {value}")

ValueError: No trials are completed yet.

In [ ]:
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values("value", ascending=False).reset_index(drop=True)
trials_df


,number,value,datetime_start,datetime_complete,duration,params_K_tournament,params_budget_penalty_multiplier,params_calorie_penalty_weight,params_crossover_probability,params_crossover_type,params_keep_elitism,params_mutation_probability,params_mutation_type,params_num_parents_mating,params_pantry_score_weight,params_parent_selection_type,params_population_size,params_protein_penalty_weight,params_waste_penalty_multiplier,state
0,49,4.136798,2026-04-26 15:26:57.473198,2026-04-26 15:27:01.052359,0 days 00:00:03.579161,4,3.5,0.001014,0.8,single_point,1,0.11,inversion,86,1.8,sss,140,0.020916,0.000169,COMPLETE
1,43,-2.757689,2026-04-26 15:26:41.454136,2026-04-26 15:26:44.418521,0 days 00:00:02.964385,4,0.5,0.001294,0.8,single_point,1,0.14,inversion,98,1.6,tournament,120,0.020588,0.000128,COMPLETE
2,42,-3.117792,2026-04-26 15:26:37.881862,2026-04-26 15:26:41.453627,0 days 00:00:03.571765,5,0.5,0.001317,0.9,single_point,1,0.15,inversion,118,1.7,tournament,140,0.022229,0.000137,COMPLETE
3,41,-3.630803,2026-04-26 15:26:34.363173,2026-04-26 15:26:37.881862,0 days 00:00:03.518689,6,0.5,0.002352,1.0,single_point,1,0.15,inversion,118,1.7,tournament,140,0.013966,0.000177,COMPLETE
4,44,-4.966042,2026-04-26 15:26:44.418521,2026-04-26 15:26:47.384459,0 days 00:00:02.965938,4,5.0,0.001268,0.8,single_point,1,0.14,inversion,98,1.6,sss,120,0.045125,0.000128,COMPLETE
5,37,-5.004788,2026-04-26 15:26:23.401903,2026-04-26 15:26:26.601008,0 days 00:00:03.199105,10,2.0,0.002414,0.8,single_point,1,0.24,inversion,98,1.7,tournament,120,0.023377,0.000200,COMPLETE
6,45,-5.054914,2026-04-26 15:26:47.384963,2026-04-26 15:26:50.419407,0 days 00:00:03.034444,4,4.5,0.001255,0.8,single_point,1,0.06,inversion,98,1.0,sss,120,0.045996,0.000128,COMPLETE
7,10,-5.182124,2026-04-26 15:24:05.625772,2026-04-26 15:24:08.327175,0 days 00:00:02.701403,7,0.5,0.001285,1.0,scattered,0,0.20,scramble,122,1.6,tournament,140,0.010586,0.000113,COMPLETE
8,28,-5.428671,2026-04-26 15:25:51.303133,2026-04-26 15:25:54.726517,0 days 00:00:03.423384,8,0.5,0.002190,1.0,two_points,0,0.18,scramble,150,1.7,tournament,180,0.013982,0.000140,COMPLETE
9,22,-5.833996,2026-04-26 15:25:35.679695,2026-04-26 15:25:38.763501,0 days 00:00:03.083806,7,1.0,0.001696,1.0,scattered,0,0.20,scramble,118,1.7,tournament,160,0.014942,0.000235,COMPLETE


In [ ]:
plot_optimization_history(study).show()

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

In [ ]:
plot_param_importances(study).show()

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

In [ ]:
plot_parallel_coordinate(study).show()